### Here we are going to normalize the view count and add it to the results of the strategies so we will be able to measure from Player's F side how good a question is without considering the time it was pulished 

In [3]:
import pandas as pd

In [ ]:
SUBJECTS = ['united']
LLMS = [
"EleutherAI-pythia-6.9b",
"meta-llama-Llama-3.1-8B",
"meta-llama-Meta-Llama-3-8B-Instruct"
]
STRATEGIES = [
"G-Greedy",
"G-Utility",
"Random"
]
samples = range(1, 51)
weeks=range(1,54)
path=f"""bootstrap_dataset/bootstrap_dataset_{{}}_{{}}_sample_{{}}_week_{{}}.parquet"""

In [ ]:
for subject in SUBJECTS:
    for llm in LLMS:
        for sample in samples:
            for week in weeks:
                file_path = path.format(subject, llm, sample, week)
                df=pd.read_parquet(file_path)[['QuestionId','ViewCount']]
                sum_view_count = df['ViewCount'].sum()
                df['NormalizedViewCount'] = df['ViewCount'] / sum_view_count
                for strategy in STRATEGIES:
                    df_strategy=pd.read_parquet(f'bootstrap_results_{strategy}/weekly_data_week_{week}_sample_{sample}_subject_{subject}_{llm}.parquet')
                    df_strategy = df_strategy.merge(df[['QuestionId', 'NormalizedViewCount']], on='QuestionId', how='left')
                    df_strategy = df_strategy[['text', 'perplexity','ViewCount','CreationDate','AnswerCount','QuestionId','year_week','NormalizedViewCount']].copy().drop_duplicates().reset_index(drop=True)
                    df_strategy.to_parquet(f'bootstrap_results_{strategy}/weekly_data_week_{week}_sample_{sample}_subject_{subject}_{llm}.parquet', index=False)